# Multi-Route Model with Low-Volume Airline-Routes Excluded

Based on findings from notebook 6b, low-volume airline-routes have weak lag1 correlation and produce unreliable predictions. This notebook trains the multi-route model after excluding these routes from the training dataset. The threshold of 50 flights/month was found to be a good balance between removing volatile routes and preserving as much training data as possible. The performance of the new model is compared against notebook 6.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

# XGBoost for classification
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)

# Create unique identifier for each airline-route combination
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']

# Sort for proper lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

## 2. Identify and Exclude Low-Volume Airline-Routes

In [ ]:
# Calculate average volume per airline-route
volume_threshold = 50  # From notebook 6b

airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']

# Identify low-volume airline-routes
low_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] < volume_threshold]['airline_route'].tolist()
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

print(f"Volume threshold: {volume_threshold} flights/month")
print(f"\nLow-volume airline-routes (<{volume_threshold} flights/month): {len(low_volume_ar)}")
print(f"High-volume airline-routes (≥{volume_threshold} flights/month): {len(high_volume_ar)}")

print("\nLow-volume airline-routes to exclude:")
print("-" * 60)
for ar in sorted(low_volume_ar):
    vol = airline_route_volume[airline_route_volume['airline_route'] == ar]['avg_volume'].values[0]
    print(f"  {ar:<45} {vol:>6.1f} flights/mo")

In [ ]:
# Filter to high-volume airline-routes only
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

print(f"\nRecords before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")
print(f"Records excluded:         {len(df) - len(df_filtered)} ({(len(df) - len(df_filtered))/len(df)*100:.1f}%)")

print(f"\nRemaining routes: {df_filtered['route'].nunique()}")
print(f"Remaining airlines: {df_filtered['airline'].nunique()}")
print(f"Remaining airline-routes: {df_filtered['airline_route'].nunique()}")

In [ ]:
# Show remaining airline-routes by route
print("Remaining airline-routes by route:")
print("=" * 60)
for route in sorted(df_filtered['route'].unique()):
    route_data = df_filtered[df_filtered['route'] == route]
    airlines = route_data['airline'].unique()
    print(f"\n{route}:")
    for airline in sorted(airlines):
        ar = f"{airline}_{route}"
        vol = airline_route_volume[airline_route_volume['airline_route'] == ar]['avg_volume'].values[0]
        print(f"  {airline:<25} {vol:>6.1f} flights/mo")

## 3. Feature Engineering

In [ ]:
# Create lagged features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)

# Create momentum feature
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Create cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encode airline
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

# One-hot encode route
route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Create exponential transformation of rainy_days_arr
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())

# Create temperature volatility total
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())

print(f"Airlines: {df_filtered['airline'].nunique()}")
print(f"Airline columns: {airline_cols}")
print(f"\nRoutes: {df_filtered['route'].nunique()}")
print(f"Route columns: {route_cols}")

# Drop rows with missing lag values
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"\nRows after dropping NaN: {len(df_clean)}")

## 4. Train/Validation/Test Split

In [ ]:
# Time-based split (excluding 2020-2022 COVID period)
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2023))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")
print("\nNote: 2020-2022 excluded (COVID period)")

# Compare with notebook 6 (full multi-route)
print("\n" + "="*60)
print("Comparison with notebook 6 (full multi-route):")
print(f"  Notebook 6 train:  2327 samples")
print(f"  Notebook 6d train: {train_mask.sum()} samples ({train_mask.sum()/2327*100:.1f}%)")

In [ ]:
# Define feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']

# Regression features
reg_features = base_features + ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp']

# Classification features
clf_features = base_features + ['rainy_days_arr', 'delay_rate_gradient', 'temp_volatility_total_exp']

print(f"Base features: {len(base_features)}")
print(f"Regression features: {len(reg_features)}")
print(f"Classification features: {len(clf_features)}")

In [ ]:
# Prepare target variables
y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

print("Target variables prepared.")
print(f"\nTest set class distribution:")
print(f"  is_high_delay=0: {(y_test_clf == 0).sum()}")
print(f"  is_high_delay=1: {(y_test_clf == 1).sum()}")

## 5. Regression Models

In [ ]:
# Train regression models
reg_results = []

# Prepare data
X_train = df_clean.loc[train_mask, reg_features].values
X_val = df_clean.loc[val_mask, reg_features].values
X_test = df_clean.loc[test_mask, reg_features].values

# Scale for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("="*60)
print("REGRESSION MODELS (High-Volume Only)")
print("="*60)

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train_reg)
ridge_test_pred = ridge.predict(X_test_scaled)

ridge_r2 = r2_score(y_test_reg, ridge_test_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
ridge_mae = mean_absolute_error(y_test_reg, ridge_test_pred)
print(f"\nRidge:    R²={ridge_r2:.4f}, RMSE={ridge_rmse:.4f}, MAE={ridge_mae:.4f}")

reg_results.append({
    'Model': 'Ridge',
    'Test_R2': ridge_r2,
    'Test_RMSE': ridge_rmse,
    'Test_MAE': ridge_mae
})

# Random Forest Regression
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train_reg)
rf_test_pred = rf_reg.predict(X_test)

rf_r2 = r2_score(y_test_reg, rf_test_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
rf_mae = mean_absolute_error(y_test_reg, rf_test_pred)
print(f"RF:       R²={rf_r2:.4f}, RMSE={rf_rmse:.4f}, MAE={rf_mae:.4f}")

reg_results.append({
    'Model': 'Random Forest',
    'Test_R2': rf_r2,
    'Test_RMSE': rf_rmse,
    'Test_MAE': rf_mae
})

reg_df = pd.DataFrame(reg_results)

## 6. Classification Models

In [ ]:
# Train classification models
clf_results = []

# Prepare data
X_train_clf = df_clean.loc[train_mask, clf_features].values
X_val_clf = df_clean.loc[val_mask, clf_features].values
X_test_clf = df_clean.loc[test_mask, clf_features].values

# Scale for linear models
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_val_clf_scaled = scaler_clf.transform(X_val_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

print("="*60)
print("CLASSIFICATION MODELS (High-Volume Only)")
print("="*60)

# Logistic Regression
logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
logreg.fit(X_train_clf_scaled, y_train_clf)
logreg_test_pred = logreg.predict(X_test_clf_scaled)
logreg_test_proba = logreg.predict_proba(X_test_clf_scaled)[:, 1]

logreg_f1 = f1_score(y_test_clf, logreg_test_pred)
logreg_auc = roc_auc_score(y_test_clf, logreg_test_proba)
logreg_acc = accuracy_score(y_test_clf, logreg_test_pred)
print(f"\nLogistic: F1={logreg_f1:.4f}, AUC={logreg_auc:.4f}, Acc={logreg_acc:.4f}")

clf_results.append({
    'Model': 'Logistic',
    'Test_F1': logreg_f1,
    'Test_AUC': logreg_auc,
    'Test_Acc': logreg_acc
})

# Random Forest Classification
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_clf, y_train_clf)
rf_clf_test_pred = rf_clf.predict(X_test_clf)
rf_clf_test_proba = rf_clf.predict_proba(X_test_clf)[:, 1]

rf_clf_f1 = f1_score(y_test_clf, rf_clf_test_pred)
rf_clf_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
rf_clf_acc = accuracy_score(y_test_clf, rf_clf_test_pred)
print(f"RF Clf:   F1={rf_clf_f1:.4f}, AUC={rf_clf_auc:.4f}, Acc={rf_clf_acc:.4f}")

clf_results.append({
    'Model': 'Random Forest',
    'Test_F1': rf_clf_f1,
    'Test_AUC': rf_clf_auc,
    'Test_Acc': rf_clf_acc
})

# XGBoost Classification
if HAS_XGB:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        min_child_weight=5, random_state=42, n_jobs=-1
    )
    xgb_clf.fit(X_train_clf, y_train_clf, eval_set=[(X_val_clf, y_val_clf)], verbose=False)
    xgb_test_pred = xgb_clf.predict(X_test_clf)
    xgb_test_proba = xgb_clf.predict_proba(X_test_clf)[:, 1]
    
    xgb_f1 = f1_score(y_test_clf, xgb_test_pred)
    xgb_auc = roc_auc_score(y_test_clf, xgb_test_proba)
    xgb_acc = accuracy_score(y_test_clf, xgb_test_pred)
    print(f"XGBoost:  F1={xgb_f1:.4f}, AUC={xgb_auc:.4f}, Acc={xgb_acc:.4f}")
    
    clf_results.append({
        'Model': 'XGBoost',
        'Test_F1': xgb_f1,
        'Test_AUC': xgb_auc,
        'Test_Acc': xgb_acc
    })

clf_df = pd.DataFrame(clf_results)

## 7. Comparison with Notebook 6 (Full Multi-Route)

In [ ]:
# Reference values from notebook 6 (full multi-route model)
ref_nb6 = {
    'Ridge': {'R2': 0.3628, 'RMSE': 0.0985},
    'Random Forest': {'R2': 0.3576, 'RMSE': 0.0989},
    'Logistic': {'F1': 0.7359, 'AUC': 0.8474},
    'Random Forest_clf': {'F1': 0.6911, 'AUC': 0.8253},
    'XGBoost': {'F1': 0.7213, 'AUC': 0.8463}
}

# Reference values from notebook 4e (single-route SYD-MEL)
ref_4e = {
    'Ridge': {'R2': 0.4941, 'RMSE': 0.0703},
    'Random Forest': {'R2': 0.4715, 'RMSE': 0.0719},
    'Logistic': {'F1': 0.7746, 'AUC': 0.8852},
    'Random Forest_clf': {'F1': 0.7514, 'AUC': 0.8555},
    'XGBoost': {'F1': 0.8065, 'AUC': 0.8742}
}

print("="*80)
print("COMPARISON: Notebook 6 (Full) vs 6d (High-Volume Only)")
print("="*80)

print("\nREGRESSION MODELS:")
print("-" * 80)
print(f"{'Model':<15} {'6 (Full) R²':>12} {'6d (Filtered) R²':>18} {'Δ R²':>10} {'Status':>12}")
print("-" * 80)
for _, row in reg_df.iterrows():
    model = row['Model']
    ref_r2 = ref_nb6[model]['R2']
    new_r2 = row['Test_R2']
    diff = new_r2 - ref_r2
    sign = '+' if diff > 0 else ''
    status = "IMPROVED" if diff > 0.01 else ("WORSE" if diff < -0.01 else "~same")
    print(f"{model:<15} {ref_r2:>12.4f} {new_r2:>18.4f} {sign}{diff:>9.4f} {status:>12}")

print("\nCLASSIFICATION MODELS:")
print("-" * 80)
print(f"{'Model':<15} {'6 (Full) F1':>12} {'6d (Filtered) F1':>18} {'Δ F1':>10} {'Status':>12}")
print("-" * 80)
for _, row in clf_df.iterrows():
    model = row['Model']
    if model == 'Random Forest':
        ref_f1 = ref_nb6['Random Forest_clf']['F1']
    else:
        ref_f1 = ref_nb6[model]['F1']
    new_f1 = row['Test_F1']
    diff = new_f1 - ref_f1
    sign = '+' if diff > 0 else ''
    status = "IMPROVED" if diff > 0.01 else ("WORSE" if diff < -0.01 else "~same")
    print(f"{model:<15} {ref_f1:>12.4f} {new_f1:>18.4f} {sign}{diff:>9.4f} {status:>12}")

In [ ]:
# Visualization: Comparison bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette("Set1",3)

# Regression R² comparison
ax = axes[0]
models_reg = reg_df['Model'].tolist()
x = np.arange(len(models_reg))
width = 0.25

single_r2 = [ref_4e[m]['R2'] for m in models_reg]
full_r2 = [ref_nb6[m]['R2'] for m in models_reg]
filtered_r2 = reg_df['Test_R2'].tolist()

ax.bar(x - width, single_r2, width, label='4e (Single-Route)', color=palette[0], alpha=0.8)
ax.bar(x, full_r2, width, label='6 (Full Multi-Route)', color=palette[1], alpha=0.8)
ax.bar(x + width, filtered_r2, width, label='6d (High-Volume Only)', color=palette[2], alpha=0.8)
ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Regression: $R^2$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models_reg)
ax.legend()
ax.set_ylim(0, 0.7)

# Classification F1 comparison
ax = axes[1]
models_clf = clf_df['Model'].tolist()
x = np.arange(len(models_clf))

single_f1 = [ref_4e[m if m != 'Random Forest' else 'Random Forest_clf']['F1'] for m in models_clf]
full_f1 = [ref_nb6[m if m != 'Random Forest' else 'Random Forest_clf']['F1'] for m in models_clf]
filtered_f1 = clf_df['Test_F1'].tolist()

ax.bar(x - width, single_f1, width, label='4e (Single-Route)', color=palette[0], alpha=0.8)
ax.bar(x, full_f1, width, label='6 (Full Multi-Route)', color=palette[1], alpha=0.8)
ax.bar(x + width, filtered_f1, width, label='6d (High-Volume Only)', color=palette[2], alpha=0.8)
ax.set_ylabel(r'Test $F_1$')
ax.set_title(r'Classification: $F_1$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models_clf)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 8. Performance by Route

In [ ]:
# Analyze performance by route
test_df = df_clean[test_mask].copy()
test_df['ridge_pred'] = ridge.predict(X_test_scaled)

# Reference values from notebook 6 (by route)
ref_nb6_by_route = {
    'Hobart_Melbourne': 0.5459,
    'Sydney_Melbourne': 0.5166,
    'Melbourne_Sydney': 0.3491,
    'Sydney_Hobart': 0.3750,
    'Hobart_Sydney': 0.1790,
    'Melbourne_Hobart': 0.0796
}

print("Performance by Route (Ridge Regression):")
print("-" * 80)
print(f"{'Route':<25} {'N':>6} {'6d R²':>10} {'6 R²':>10} {'Δ':>10} {'Status':>12}")
print("-" * 80)

route_metrics = []
for route in sorted(test_df['route'].unique()):
    route_data = test_df[test_df['route'] == route]
    y_true = route_data['delay_rate']
    y_pred = route_data['ridge_pred']
    
    r2 = r2_score(y_true, y_pred)
    ref_r2 = ref_nb6_by_route.get(route, 0)
    diff = r2 - ref_r2
    sign = '+' if diff > 0 else ''
    status = "IMPROVED" if diff > 0.02 else ("WORSE" if diff < -0.02 else "~same")
    
    route_metrics.append({
        'Route': route,
        'N': len(route_data),
        'R2': r2,
        'Ref_R2': ref_r2,
        'Diff': diff
    })
    print(f"{route:<25} {len(route_data):>6} {r2:>10.4f} {ref_r2:>10.4f} {sign}{diff:>9.4f} {status:>12}")

route_metrics_df = pd.DataFrame(route_metrics)

In [ ]:
# Visualize R² by route: 6 vs 6d comparison
fig, ax = plt.subplots(figsize=(12, 6))

routes = route_metrics_df['Route'].tolist()
x = np.arange(len(routes))
width = 0.35

r2_6 = [ref_nb6_by_route.get(r, 0) for r in routes]
r2_6d = route_metrics_df['R2'].tolist()

ax.bar(x - width/2, r2_6, width, label='6 (Full Multi-Route)', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, r2_6d, width, label='6d (High-Volume Only)', color='#27ae60', alpha=0.8)

ax.set_ylabel(r'Test $R^2$')
ax.set_title('Ridge R² by Route: Notebook 6 vs 6d')
ax.set_xticks(x)
ax.set_xticklabels([r.replace('_', '→') for r in routes], rotation=45, ha='right')
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Summary and Observations

In [ ]:
print("="*80)
print("SUMMARY: Impact of Excluding Low-Volume Airline-Routes")
print("="*80)

print("\nDataset Comparison:")
print("-" * 60)
print(f"  {'Metric':<30} {'6 (Full)':>15} {'6d (Filtered)':>15}")
print("-" * 60)
print(f"  {'Airline-routes':<30} {'35':>15} {len(high_volume_ar):>15}")
print(f"  {'Training samples':<30} {'2327':>15} {train_mask.sum():>15}")
print(f"  {'Test samples':<30} {'510':>15} {test_mask.sum():>15}")
print(f"  {'Volume threshold':<30} {'None':>15} {f'≥{volume_threshold} flights/mo':>15}")

print("\n" + "="*80)
print("PERFORMANCE IMPROVEMENT")
print("="*80)

# Calculate improvements
reg_improvements = []
for _, row in reg_df.iterrows():
    model = row['Model']
    ref_r2 = ref_nb6[model]['R2']
    new_r2 = row['Test_R2']
    reg_improvements.append(new_r2 - ref_r2)

clf_improvements = []
for _, row in clf_df.iterrows():
    model = row['Model']
    if model == 'Random Forest':
        ref_f1 = ref_nb6['Random Forest_clf']['F1']
    else:
        ref_f1 = ref_nb6[model]['F1']
    new_f1 = row['Test_F1']
    clf_improvements.append(new_f1 - ref_f1)

print(f"\nRegression (R²):")
for i, (_, row) in enumerate(reg_df.iterrows()):
    sign = '+' if reg_improvements[i] > 0 else ''
    print(f"  {row['Model']:<20}: {ref_nb6[row['Model']]['R2']:.4f} → {row['Test_R2']:.4f} ({sign}{reg_improvements[i]:.4f})")

print(f"\nClassification (F1):")
for i, (_, row) in enumerate(clf_df.iterrows()):
    sign = '+' if clf_improvements[i] > 0 else ''
    model = row['Model']
    if model == 'Random Forest':
        ref = ref_nb6['Random Forest_clf']['F1']
    else:
        ref = ref_nb6[model]['F1']
    print(f"  {model:<20}: {ref:.4f} → {row['Test_F1']:.4f} ({sign}{clf_improvements[i]:.4f})")

### Observations

- The performance of regression models noticeably **improved**:
   - Average R² change: +0.0944
   - Ridge: from 0.3628 up to 0.4487
   - Random Forest: from 0.3576 to 0.4605

- The classification models are also improved:
   - Average F1 change across all models: +0.0299

- While filtering out the low-volume airline-routes improved overall performance, it sacrifices the capability to predict low-volume routes, which is acceptable given their inherent volatility.

- Based on these consideration, the low-volume route filtering will be included in the training data preparation.

## 10. Next Step

Having established that the model scales well to additional cities, subsequent notebooks evaluate further feature additions.